# Lab Assignment 1: Neural Network Implementation

**Name:** Parimal Ahire  
**PRN:** 202301040067  
**Course:** Deep Learning Lab  
**GitHub Repository:** https://github.com/ParimalAhire/Lab-Assignment-1-Neural-Network


## Aim
To implement a feedforward neural network in three ways:
1. **From Scratch** using NumPy (forward pass, backpropagation, gradient descent)
2. **Using Keras** (high-level deep learning library)
3. **Using Scikit-learn** (MLPClassifier)

The network learns five logical gate functions **(XOR, AND, OR, NAND, NOR)** simultaneously and results are compared across all three approaches.


## Algorithm
1. Import all required libraries once
2. Define shared dataset (inputs X and targets y for 5 logic gates)
3. **Part 1:** Build ANN from scratch — forward pass → loss → backprop → weight update
4. **Part 2:** Build ANN using Keras Sequential API
5. **Part 3:** Build ANN using Scikit-learn MLPClassifier
6. Compare predictions and accuracy across all three approaches
7. Single combined user input test — all three models output together


---
## Step 1: Import All Libraries
> All imports are done **once here** and shared across all three parts.


In [ ]:
# Core
import numpy as np
import matplotlib.pyplot as plt

# Keras / TensorFlow
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

# Scikit-learn
from sklearn.neural_network import MLPClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print('All libraries imported successfully!')
print('Keras version:', keras.__version__)


## Step 2: Define Shared Dataset
> The same `X`, `y` and `gate_names` are used by **all three parts**.  
> To change the dataset, **only edit this cell** — everything else updates automatically.

Each row in `X` → one input combination: `[0,0]`, `[0,1]`, `[1,0]`, `[1,1]`  
Each column in `y` → one gate output: **XOR, AND, OR, NAND, NOR**


In [ ]:
# Input: all 4 combinations of 2 binary inputs
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]], dtype=float)

# Output: 5 logic gates — XOR, AND, OR, NAND, NOR
y = np.array([[0, 0, 0, 1, 1],
              [1, 0, 1, 1, 0],
              [1, 0, 1, 1, 0],
              [0, 1, 1, 0, 0]], dtype=float)

gate_names = ['XOR', 'AND', 'OR', 'NAND', 'NOR']

# Display truth table
print('Truth Table:')
print(f"{'Input':^12}", end='')
for g in gate_names:
    print(f'{g:^7}', end='')
print()
print('-' * 47)
for i in range(4):
    print(f"{str(X[i].astype(int).tolist()):^12}", end='')
    for j in range(5):
        print(f"{int(y[i,j]):^7}", end='')
    print()


---
# Part 1: ANN From Scratch (NumPy Only)
---

## Step 3: Define Activation Function

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    """Derivative of sigmoid given its output (not input)"""
    return x * (1 - x)


## Step 4: Network Configuration & Weight Initialization

In [ ]:
# Hyperparameters
input_neurons  = 2
hidden_neurons = 5
output_neurons = 5
learning_rate  = 0.5
epochs         = 1000

# Random weight initialization
W1 = np.random.randn(input_neurons, hidden_neurons)
b1 = np.zeros((1, hidden_neurons))
W2 = np.random.randn(hidden_neurons, output_neurons)
b2 = np.zeros((1, output_neurons))

print(f'Architecture : {input_neurons} -> {hidden_neurons} -> {output_neurons}')
print(f'Learning Rate: {learning_rate} | Epochs: {epochs}')
print('\nInitial W1:\n', W1)
print('\nInitial W2:\n', W2)


## Step 5: Training (Forward Pass + Backpropagation + Gradient Descent)
> Weight updates are printed every **100 epochs** for readability.


In [ ]:
losses_scratch = []

for epoch in range(epochs):

    # ---- Forward Pass ----
    z1 = np.dot(X, W1) + b1
    a1 = sigmoid(z1)
    z2 = np.dot(a1, W2) + b2
    a2 = sigmoid(z2)

    # ---- Loss (MSE) ----
    loss = np.mean((a2 - y) ** 2)
    losses_scratch.append(loss)

    # ---- Backpropagation ----
    d_output = (a2 - y) * sigmoid_derivative(a2)
    d_hidden = np.dot(d_output, W2.T) * sigmoid_derivative(a1)

    # ---- Gradients ----
    dW2 = np.dot(a1.T, d_output)
    db2 = np.sum(d_output, axis=0, keepdims=True)
    dW1 = np.dot(X.T, d_hidden)
    db1 = np.sum(d_hidden, axis=0, keepdims=True)

    # ---- Weight Update (Gradient Descent) ----
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1

    # Print every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1:4d} | Loss: {loss:.6f}')
        print(f'  Updated W1:\n{np.round(W1, 4)}')
        print(f'  Updated W2:\n{np.round(W2, 4)}')
        print()


## Step 6: Final Predictions & Loss Curve (From Scratch)

In [ ]:
# Forward pass with final weights
z1 = np.dot(X, W1) + b1
a1 = sigmoid(z1)
z2 = np.dot(a1, W2) + b2
predictions_scratch = sigmoid(z2)

print('Predictions vs Expected (predicted → rounded):')
print(f"{'Input':^12}", end='')
for g in gate_names:
    print(f'{g:^12}', end='')
print()
print('-' * 72)
for i in range(4):
    print(f"{str(X[i].astype(int).tolist()):^12}", end='')
    for j in range(5):
        pred  = round(predictions_scratch[i, j], 2)
        exp   = int(y[i, j])
        print(f"{str(pred)+'('+str(exp)+')':^12}", end='')
    print()
print('\nFinal Loss:', round(losses_scratch[-1], 6))

# Loss curve
plt.figure(figsize=(8, 4))
plt.plot(losses_scratch, color='blue', linewidth=2)
plt.title('Training Loss Over Epochs (From Scratch)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True)
plt.tight_layout()
plt.show()


---
# Part 2: ANN Using Keras
---

## Step 7: Build & Train Keras Model

- **Architecture:** 2 → 5 (hidden, sigmoid) → 5 (output, sigmoid)
- **Optimizer:** Adam (adaptive learning rate — faster than plain gradient descent)
- **Loss:** Mean Squared Error


In [ ]:
model_keras = Sequential([
    Dense(5, input_dim=2, activation='sigmoid', name='hidden_layer'),
    Dense(5, activation='sigmoid', name='output_layer')
], name='ANN_Keras')

model_keras.compile(
    optimizer=Adam(learning_rate=0.05),
    loss='mean_squared_error'
)

model_keras.summary()

history_keras = model_keras.fit(
    X, y,
    epochs=1000,
    batch_size=4,
    verbose=0
)

print('Training complete!')
print(f'Final Loss: {history_keras.history["loss"][-1]:.6f}')


## Step 8: Final Predictions & Loss Curve (Keras)

In [ ]:
predictions_keras = model_keras.predict(X, verbose=0)

print('Keras Predictions vs Expected (predicted → rounded):')
print(f"{'Input':^12}", end='')
for g in gate_names:
    print(f'{g:^12}', end='')
print()
print('-' * 72)
for i in range(4):
    print(f"{str(X[i].astype(int).tolist()):^12}", end='')
    for j in range(5):
        pred  = round(predictions_keras[i, j], 2)
        exp   = int(y[i, j])
        print(f"{str(pred)+'('+str(exp)+')':^12}", end='')
    print()
print('\nFinal Loss:', round(history_keras.history['loss'][-1], 6))

# Loss curve
plt.figure(figsize=(8, 4))
plt.plot(history_keras.history['loss'], color='green', linewidth=2)
plt.title('Training Loss Over Epochs (Keras)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.grid(True)
plt.tight_layout()
plt.show()


---
# Part 3: ANN Using Scikit-learn (MLPClassifier)
---

## Step 9: Build & Train MLPClassifier

- **Solver:** `adam` (consistent with Keras; `lbfgs` is also suitable for small datasets)
- **MultiOutputClassifier** wraps MLP to handle all 5 gate outputs at once


In [ ]:
base_mlp = MLPClassifier(
    hidden_layer_sizes=(5,),      # 1 hidden layer, 5 neurons
    activation='logistic',        # sigmoid
    solver='adam',
    learning_rate_init=0.05,
    max_iter=10000,
    random_state=42
)

model_sklearn = MultiOutputClassifier(base_mlp)
model_sklearn.fit(X, y.astype(int))

print('Training complete!')


## Step 10: Final Predictions & Accuracy (Sklearn)

In [ ]:
predictions_sklearn = model_sklearn.predict(X)

print('Sklearn Predictions vs Expected (predicted → rounded):')
print(f"{'Input':^12}", end='')
for g in gate_names:
    print(f'{g:^12}', end='')
print()
print('-' * 72)
for i in range(4):
    print(f"{str(X[i].astype(int).tolist()):^12}", end='')
    for j in range(5):
        pred = predictions_sklearn[i, j]
        exp  = int(y[i, j])
        print(f"{str(pred)+'('+str(exp)+')':^12}", end='')
    print()

print('\nAccuracy per Gate:')
for j, gate in enumerate(gate_names):
    acc = accuracy_score(y[:, j].astype(int), predictions_sklearn[:, j])
    print(f'  {gate:5s}: {acc * 100:.1f}%')


---
# Part 4: Comparison of All Three Approaches
---

## Step 11: Side-by-Side Prediction Table & Accuracy

In [ ]:
# Binarize all predictions
pred_scratch = (predictions_scratch >= 0.5).astype(int)
pred_keras   = (predictions_keras   >= 0.5).astype(int)
pred_sklearn = predictions_sklearn.astype(int)
y_int        = y.astype(int)

print('=' * 70)
print(f"{'Full Comparison Table':^70}")
print('=' * 70)
print(f"{'Input':^8} {'Gate':^6} {'Expected':^10} {'Scratch':^10} {'Keras':^10} {'Sklearn':^10}")
print('-' * 70)
for i in range(4):
    inp = str(X[i].astype(int).tolist())
    for j, gate in enumerate(gate_names):
        print(f"{inp:^8} {gate:^6} {y_int[i,j]:^10} "
              f"{pred_scratch[i,j]:^10} {pred_keras[i,j]:^10} {pred_sklearn[i,j]:^10}")
    print('-' * 70)

def overall_acc(pred, true):
    return np.mean(pred == true) * 100

print('\nOverall Accuracy:')
print(f'  From Scratch : {overall_acc(pred_scratch, y_int):.1f}%')
print(f'  Keras        : {overall_acc(pred_keras,   y_int):.1f}%')
print(f'  Sklearn      : {overall_acc(pred_sklearn, y_int):.1f}%')


## Step 12: Combined Loss Curve (Scratch vs Keras)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(losses_scratch,                 color='blue',  linewidth=2, label='From Scratch')
plt.plot(history_keras.history['loss'],  color='green', linewidth=2, label='Keras')
plt.title('Training Loss Comparison: From Scratch vs Keras')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


---
## Step 13: Test All Three Models — Single User Input
> Enter input **once** and see outputs from all three models side by side.
---

In [ ]:
a = float(input('Enter first input  (0 or 1): '))
b = float(input('Enter second input (0 or 1): '))

test_input = np.array([[a, b]])

# --- From Scratch ---
z1_t = np.dot(test_input, W1) + b1
a1_t = sigmoid(z1_t)
z2_t = np.dot(a1_t, W2) + b2
out_scratch = sigmoid(z2_t)[0]

# --- Keras ---
out_keras = model_keras.predict(test_input, verbose=0)[0]

# --- Sklearn ---
out_sklearn = model_sklearn.predict(test_input)[0]
out_proba   = model_sklearn.predict_proba(test_input)

# --- Display ---
print(f'\nInput: ({int(a)}, {int(b)})')
print()
print(f"{'Gate':<8} {'Scratch':^16} {'Keras':^16} {'Sklearn':^16}")
print('-' * 60)
for i, gate in enumerate(gate_names):
    s_val   = round(out_scratch[i], 2)
    s_label = 1 if s_val >= 0.5 else 0

    k_val   = round(out_keras[i], 2)
    k_label = 1 if k_val >= 0.5 else 0

    sk_prob  = round(out_proba[i][0][1], 2)
    sk_label = out_sklearn[i]

    print(f"{gate:<8} {str(s_val)+'→'+str(s_label):^16} "
          f"{str(k_val)+'→'+str(k_label):^16} "
          f"{str(sk_prob)+'→'+str(sk_label):^16}")
print('-' * 60)
print('(format: probability → predicted label)')


## Summary Comparison

| Feature | From Scratch | Keras | Scikit-learn |
|---|---|---|---|
| Library | NumPy | TensorFlow/Keras | scikit-learn |
| Optimizer | Gradient Descent | Adam | Adam / lbfgs |
| Backprop | Manual (coded) | Automatic | Automatic |
| Code complexity | High | Low | Low |
| Transparency | Full control | Black box | Black box |
| Best for | Learning concepts | Deep learning | Quick ML baselines |


## Applications

- **Image recognition** – Facial recognition, object detection, medical imaging
- **Speech processing** – Voice assistants, speech-to-text systems
- **Fraud detection** – Identifying suspicious transactions
- **Medical diagnosis** – Disease prediction from patient data
- **Recommendation systems** – Netflix, YouTube, Amazon suggestions
- **Autonomous systems** – Self-driving cars, robotics


## Conclusion

This assignment implemented a feedforward neural network in three ways to learn five logic gates simultaneously:

1. **From Scratch (NumPy):** Manually coded forward pass, backpropagation, and gradient descent. Provides full transparency into how weights are updated at each step.

2. **Keras:** High-level API with just a few lines of code. Adam optimizer gave faster convergence compared to plain gradient descent.

3. **Scikit-learn:** MLPClassifier with MultiOutputClassifier wrapper for multi-label output. Simplest to implement for quick baselines.

All three approaches successfully learned all five gate functions. A single user input cell at the end tests all three models together, making comparison straightforward. The exercise demonstrates how the same neural network concept can be implemented at different levels of abstraction.
